# 04 - Interpretability: what moves the recovery score

Review aid for `src/attribution.py` (run it first, then `src/m2_figures.py`).

**Method recap.** Exact grouped Shapley values, in recovery points, against ONE fixed reference B
(mean prediction over a fixed sample of 100 training cycles). For every cycle the group contributions
sum exactly to `prediction - B`; day-over-day differences sum exactly to the change in prediction.
Both models (logit-ridge "formula" and XGBoost) are fit on the training period only, exactly as evaluated.

Attribution describes the **model**, not physiology. Committed without outputs on purpose:
section 6 shows per-cycle values.

In [ ]:
import pandas as pd
from IPython.display import Image, display
pd.set_option("display.precision", 3)
summary = pd.read_csv("../results/m2_attribution_summary.csv")
meta = pd.read_csv("../results/m2_attribution_meta.csv", index_col=0)["value"]
meta

## 1. Share of attribution (how much of the score's spread each group accounts for)

In [ ]:
display(Image("../results/figures/m2_attribution_share.png"))
summary.pivot(index="group", columns="model", values="share_of_attribution").sort_values("formula", ascending=False)

## 2. Day-over-day moves (the attribution question from Appendix B)

In [ ]:
display(Image("../results/figures/m2_daily_move_share.png"))
summary.pivot(index="group", columns="model", values=["share_of_daily_moves", "top_driver_freq"]).round(3)

## 3. Stability across 25 refits (5 CV-fold training sets + 20 random 80% subsamples)
Narrow 5-95% ranges = the credit split is stable; wide = report with caution (Appendix C-7).

In [ ]:
cols = [c for c in summary.columns if c.startswith("share_of_attribution_p")]
summary.set_index(["model", "group"])[cols].sort_index()

## 4. The reconstructed formula, in readable units
Points at a typical day (median training recovery). The logit link means the same change moves the
score less near 0 or 100. Single coefficients inside correlated inputs are confounded -- read groups,
not individual rows, as the result (e.g. deep sleep % and disturbances carry odd signs).

In [ ]:
pd.read_csv("../results/m2_formula_effects.csv")

## 5. Interaction: does sleep matter less when HRV is high?

In [ ]:
display(Image("../results/figures/m2_hrv_sleep_interaction.png"))
pd.read_csv("../results/m2_interaction.csv")

## 6. Worked example (private): the largest predicted day-over-day move in the held-out period

In [ ]:
contrib = pd.read_csv("../data/processed/m2_contributions.csv")
feats = pd.read_csv("../data/processed/features.csv", usecols=["cycle_id", "cycle_start_local"])
groups = [c for c in contrib.columns if c not in
          ["model", "cycle_id", "is_test", "baseline_B", "predicted", "actual"]]
f = contrib[contrib.model == "formula"].merge(feats, on="cycle_id").reset_index(drop=True)
f["dpred"] = f["predicted"].diff()   # NB: also spans gaps; fine for picking an example
i = f[f.is_test].dpred.abs().idxmax()
print(f.loc[i - 1, "cycle_start_local"], "->", f.loc[i, "cycle_start_local"])
print("actual:", f.loc[i - 1, "actual"], "->", f.loc[i, "actual"],
      "| predicted:", round(f.loc[i - 1, "predicted"], 1), "->", round(f.loc[i, "predicted"], 1))
(f.loc[i, groups] - f.loc[i - 1, groups]).sort_values(key=abs, ascending=False).round(1)

Plan Appendix C checklist: 5 fixed baseline + differencing ✓ · 6 linear + trees with agreement ✓ ·
7 stability spread ✓ · 8 not causation (stated) · 9 baseline B reported in `meta`.